In [13]:
import os
import json
import pandas as pd
from datetime import datetime
import logging
from add_ids_to_dataset_utils import *

In [14]:
class Config:
    
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    config_path = '../configs/pretreatment_config.json'

    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            _raw_config = json.load(f)
            _raw_dataset_name = _raw_config.get("RAW_DATASET_NAME")
    except FileNotFoundError:
        logging.warning(f"配置文件未找到: {config_path}")    

    # 读取的固定文件路径
    DATASET = os.path.join(IN_DIR, _raw_dataset_name)    

    _base_name = os.path.splitext(_raw_dataset_name)[0]

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, _base_name, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, _base_name, 'deduplicate')

    IDS_MAP_TABLE = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_raw-ds_fetch-ids_add-ids.xlsx')

    # 输出的文件路径
    DATASET_WITH_IDS = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_raw-ds_add-ids_dd.xlsx')
    DATASET_WITH_IDS_OUT = os.path.join(OUT_DIR, f'{_base_name}_{TIMESTAMP}_out_ds-with-ids.xlsx')

In [15]:
file_ext = os.path.splitext(Config.DATASET)[1].lower()

try:
    if file_ext == '.csv':
        dataset = pd.read_csv(Config.DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        dataset = pd.read_excel(Config.DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")


ids_map_table = pd.read_excel(Config.IDS_MAP_TABLE)

In [16]:
component_columns = ['c1', 'c2', 'c3']
key_column = 'compound'
dataset_with_ids = add_ids_to_dataset(dataset, ids_map_table, component_columns, key_column)

In [17]:
dataset_with_ids_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, 'ds-with-ids_qlty-sum.xlsx')
data_quality_summary(dataset_with_ids, dataset_with_ids_sum_output_path)

In [18]:
# 获取所有不包含'3'的列名
columns_without_3 = [col for col in dataset_with_ids.columns if '3' not in col]

# 1. 记录原始行数
original_count = len(dataset_with_ids)

# 只对这些列进行dropna操作
dataset_with_ids_cleaned = dataset_with_ids.dropna(subset=columns_without_3)

# 2. 记录清理后的行数，并计算差值
cleaned_count = len(dataset_with_ids_cleaned)
dropped_count = original_count - cleaned_count

# 3. 打印结果
print(f"缺失值清理完成：")
print(f"  - 原始数据共 {original_count} 行")
print(f"  - 删除了 {dropped_count} 行含有空值的数据")
print(f"  - 剩余有效数据 {cleaned_count} 行")

# 保存文件
dataset_with_ids_cleaned.to_excel(Config.DATASET_WITH_IDS, index=False)
dataset_with_ids_cleaned.to_excel(Config.DATASET_WITH_IDS_OUT, index=False)

缺失值清理完成：
  - 原始数据共 22386 行
  - 删除了 1219 行含有空值的数据
  - 剩余有效数据 21167 行
